# README
This jupyter notebook holds the functionality for the handover comparison. It was used during the testing of this comparison. 
The code in this notebook is not used anymore, since it has been added to the comparison_module.py to enable the run of the complete pipeline. 

This notebook could be used to explore the handover comparison as used in the comparison module, but for cleaner and easier readable code the comparison_module.py and the utils.handover_comparison.py files are more suitable

For a reader that is not familiar with python code and would just like to execute and explore the S2CF+C approach or comparison module, I would suggest to look into the S2CF and comparison module, and the corresponding util files since they are more cleaned up and commented on.

In [ ]:
import sys

import os
import re
VERSION = 14


repo_code_path = r""

if repo_code_path not in sys.path:
    sys.path.insert(0, repo_code_path)

print("Added to sys.path:", repo_code_path)

# Input folders if phase time / file
input_folder_paths = {
}

In [ ]:
# Preprocessing Handover analysis

from pathlib import Path


ID_PATTERN = re.compile(r'\bID\s*(\d+)\b')

study_event_logs_folder = f''

# Group studys in log
for dataset_key, folder_path in input_folder_paths.items():
  output_folder = os.path.join(study_event_logs_folder, dataset_key)
  if not Path(output_folder).is_dir():
    print('First run the filter_activity_annotated_logs script')

In [ ]:
# Handover analysis


from comparison_analysis.utils.handover_comparison import count_switches_folder
from pre_processing.utils.data_refining import filter_csv_by_prefix


dataset_keys = [
  f for f in os.listdir(study_event_logs_folder)
  if os.path.isdir(os.path.join(study_event_logs_folder, f))
]

for dataset_key in dataset_keys:
  V2_annotated_folder = os.path.join(study_event_logs_folder, dataset_key, 'V2_Annotated_End_Times')
  removed_V2_folder = os.path.join(study_event_logs_folder, dataset_key, 'Removed_V2')
  
  if os.path.isdir(V2_annotated_folder):

    os.makedirs(removed_V2_folder, exist_ok=True)
    file_paths = [
      os.path.join(V2_annotated_folder, f) for f in os.listdir(V2_annotated_folder)
    ]
    
    removed_V2_file_paths = [] 

    for file_path in file_paths:
      file_name = os.path.basename(file_path)
      output_file_name = os.path.join(removed_V2_folder, file_name)
      filter_csv_by_prefix(
        input_file=file_path,
        output_file=output_file_name,
        column_name='Activity',
        pattern=r"^2_\d+[A-Za-z]*_.*"
      )
      removed_V2_file_paths.append(output_file_name)

  else:
    end_time_folder = os.path.join(study_event_logs_folder, dataset_key, 'End_Times')
    removed_V2_file_paths = [
      os.path.join(end_time_folder, f) for f in os.listdir(end_time_folder)
    ]

  swith_count, switch_per_study, _ = count_switches_folder(
    removed_V2_file_paths,
    column_name='Resource',
    add_empty=False,
  )
  print(f'switch count for {dataset_key}: {swith_count}')
  print(f'total switch count for {dataset_key}: {sum(swith_count.values())}')